# Technical Troubleshooting Conversational AI System

---

| Field | Details |
|---|---|
| **Course** | Advanced NLP Applications: Conversational AI and Sentiment Intelligence |
| **Assignment** | Assignment 2 — Problem Statement 1 |
| **Domain** | IT / Technical Troubleshooting |

---

## Problem Statement

Design and implement an advanced task-oriented conversational AI system capable of:
- Understanding user intent across multiple dialogue turns
- Maintaining and updating dialogue state as the conversation progresses
- Calling simulated external tools to complete user tasks
- Handling ambiguous, incomplete, contradictory, and unsafe inputs gracefully
- Generating safe, context-aware, and personalised responses
- Evaluating the quality of conversations across multiple dimensions

## Domain

**IT Technical Troubleshooting** was selected as the domain. Users interact with the system to report device, software, network, or account issues. The system diagnoses the problem, retrieves solutions, raises support tickets when needed, and escalates to a human agent for unresolved or critical issues.

This domain was chosen because it offers well-defined intents and entities, realistic tool simulation opportunities (knowledge base lookup, ticketing system, status checker), and a broad set of edge cases including ambiguous descriptions, missing information, and frustrated users.

## Approach

The system is built as a **hybrid conversational pipeline**:

- **Interactive mode** — the fully functional chatbot accepts free-text input from the user in real time
- **Scripted evaluation mode** — 10 pre-defined conversations run automatically through the pipeline with all outputs captured in the notebook

This ensures the system can be demonstrated live while also producing reproducible, gradeable outputs.

## Notebook Structure

| Section | Content |
|---|---|
| 1 | Environment Setup |
| 2 | Domain Design |
| 3 | Intent Detection, Entity Extraction & Dialogue State Tracking |
| 4 | Tool-Augmented Response Generation |
| 5 | Memory, Personalization, Ambiguity & Safety |
| 6 | Evaluation |
| 7 | Conclusion |

---

## 1. Environment Setup

Before building the system, we check that all required libraries are available. If any are missing, they are **automatically installed** — no manual steps are needed. This makes the notebook portable across any Python environment: local machines, cloud notebooks, or restricted lab environments.

This notebook requires the following libraries:

| Library | Purpose |
|---|---|
| `scikit-learn` | TF-IDF vectorizer, Logistic Regression classifier, evaluation metrics |
| `spacy` | Named Entity Recognition for extracting user names from free text |
| `spacy: en_core_web_sm` | Pre-trained English NLP model used by spaCy |
| `pandas` | Structured output tables throughout the notebook |
| `matplotlib` | Confusion matrix and evaluation bar charts |
| `seaborn` | Heatmap styling for the confusion matrix |

All other dependencies (`re`, `json`, `collections`, `warnings`) are part of the Python standard library and require no installation.

The setup runs in two steps:
1. **Auto-install** — detect missing packages and install them immediately
2. **Verify** — re-import everything to confirm all packages are loadable in the current kernel

In [ ]:
import sys
import subprocess
import importlib

print(f"Python version : {sys.version}")
print(f"Platform       : {sys.platform}")
print()

# ── Step 1: Auto-install missing packages ────────────────────────────────────
# Tries three install strategies in order, stopping at the first that works.
# This handles: standard environments, user-level installs, and
# system-managed Python environments (PEP 668, e.g. Homebrew/Debian).

REQUIRED = {
    "sklearn":    "scikit-learn",
    "spacy":      "spacy",
    "pandas":     "pandas",
    "matplotlib": "matplotlib",
    "seaborn":    "seaborn",
}

def try_install(package):
    """Try pip install with three fallback strategies."""
    strategies = [
        [sys.executable, "-m", "pip", "install", "--quiet", package],
        [sys.executable, "-m", "pip", "install", "--quiet", "--user", package],
        [sys.executable, "-m", "pip", "install", "--quiet",
         "--break-system-packages", package],
    ]
    for cmd in strategies:
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            return True, None
    return False, result.stderr.strip()

print("Step 1 — Checking and installing missing packages...")
print("-" * 50)

installed_any = False

for module, package in REQUIRED.items():
    try:
        importlib.import_module(module)
        print(f"  OK       {package}")
    except ImportError:
        print(f"  MISSING  {package}  -->  installing...", end=" ", flush=True)
        success, err = try_install(package)
        if success:
            print("done.")
            installed_any = True
        else:
            print(f"FAILED.")
            print(f"           Install manually: pip install {package}")

# ── Step 2: Check/download spaCy model ───────────────────────────────────────
print()
print("Step 2 — Checking spaCy language model (en_core_web_sm)...")
print("-" * 50)

try:
    import spacy
    spacy.load("en_core_web_sm")
    print("  OK       en_core_web_sm")
except OSError:
    print("  MISSING  en_core_web_sm  -->  downloading...", end=" ", flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "spacy", "download", "--quiet", "en_core_web_sm"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("done.")
        installed_any = True
    else:
        print("FAILED.")
        print("           Install manually: python -m spacy download en_core_web_sm")

# ── Step 3: Verify all packages are now importable ───────────────────────────
print()
print("Step 3 — Verifying all packages are importable...")
print("-" * 50)

all_ok = True
for module, package in REQUIRED.items():
    try:
        mod = importlib.import_module(module)
        version = getattr(mod, "__version__", "installed")
        print(f"  OK  {package:<22} version: {version}")
    except ImportError:
        print(f"  FAIL  {package}  -- could not import after install attempt")
        all_ok = False

try:
    import spacy as _spacy
    _spacy.load("en_core_web_sm")
    print(f"  OK  {'en_core_web_sm':<22} spaCy model ready")
except Exception as e:
    print(f"  FAIL  en_core_web_sm  -- {e}")
    all_ok = False

print()
print("-" * 50)
if all_ok:
    print("All dependencies are satisfied. Ready to proceed.")
    if installed_any:
        print()
        print("NOTE: New packages were installed in this session.")
        print("      If any imports fail in the next cell, restart the kernel")
        print("      (Kernel > Restart) and re-run this cell once.")
else:
    print("One or more dependencies could not be installed automatically.")
    print("Please run the following in a terminal, then restart the kernel:")
    print()
    print("  pip install scikit-learn spacy pandas matplotlib seaborn")
    print("  python -m spacy download en_core_web_sm")

**Reading the output above:**

- **Step 1** shows whether each library was already present (`OK`) or needed installation. If a package was installed, `done.` confirms it completed successfully.
- **Step 2** does the same for the spaCy language model `en_core_web_sm`, downloading it automatically if absent.
- **Step 3** re-imports everything to confirm all packages are loadable in the current kernel, and prints the version of each.

**If Step 3 shows `All dependencies are satisfied`** — proceed directly to the next cell.

**If the output says `NOTE: New packages were installed`** — restart the kernel (`Kernel → Restart`) and re-run this cell once. New packages installed mid-session are not always immediately visible to the running kernel; a restart ensures they are picked up cleanly.

**If any package shows `FAILED`** — the environment does not have internet access or pip is restricted. In that case, contact the environment administrator or run the following manually in a terminal before re-opening the notebook:
```
pip install scikit-learn spacy pandas matplotlib seaborn
python -m spacy download en_core_web_sm
```

With all dependencies confirmed, we import everything the notebook requires in a single cell. Centralising all imports here makes the notebook easier to debug — if an import fails, it fails immediately rather than mid-way through a later section.

In [ ]:
# Standard library
import re
import json
import warnings
from collections import defaultdict

# Data handling
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# NLP
import spacy

warnings.filterwarnings("ignore")

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

print("All imports successful.")
print(f"  pandas     {pd.__version__}")
print(f"  spacy      {spacy.__version__}")
print(f"  sklearn    available")

All libraries are loaded and the spaCy English model is ready. The `warnings` filter suppresses minor deprecation notices that do not affect functionality.

We now move on to designing the domain — defining the intents, slots, sample conversations, and conversation flow that the system will operate within.

---

## 2. Domain Design

A well-designed domain is the foundation of any task-oriented conversational AI system. Before writing a single line of NLU code, we must clearly define what the system is trying to do, who it is talking to, what it can understand, and what information it needs to collect.

This section covers:
- **Domain overview** — the purpose of the system and its target users
- **Intents** — the set of user goals the system can recognise
- **Entity slots** — the structured information the system needs to collect
- **Sample conversations** — three realistic multi-turn dialogues showing the system in action
- **State transition logic** — how the system decides what to do next based on current state and detected intent
- **Failure cases** — known input patterns that require special handling

### Why Dialogue Systems Require Intent Detection, Entity Extraction, Context Tracking, and State Management

A conversational AI system is fundamentally different from a search engine or a static FAQ. A user interacting with a support bot rarely states everything in a single message. They say "my laptop is broken" and then, only when asked, mention that it is a Windows 11 machine with a blue screen error. The system must be capable of understanding each fragment, connecting it to what came before, and deciding what to ask or do next.

Each component of the NLU pipeline addresses a specific challenge:

**Intent Detection** answers the question: *what does the user want right now?* Without it, the system cannot distinguish between a user reporting a new problem, asking for a fix, checking an existing ticket, or trying to end the conversation. Every turn requires a fresh intent decision because user goals shift mid-conversation.

**Entity Extraction** answers the question: *what specific information did the user provide?* A sentence like "my Outlook keeps crashing on Windows 10" contains three slot values — `app_name = Outlook`, `issue_category = software`, `os = Windows 10`. Without extraction, this information is lost in unstructured text and cannot drive any action.

**Context Tracking** answers the question: *what has the user already told me?* A bot that re-asks for the device type after the user already mentioned it is not only inefficient — it signals to the user that the system is not listening. Context tracking ensures information given in turn 2 is still available in turn 6.

**State Management** answers the question: *what should the system do next given everything it knows?* The same intent can lead to different actions depending on the current state — a `provide_info` utterance means "ask the next question" if slots are still missing, but "retrieve a solution" if all required slots are now filled. State management is what makes the system goal-directed rather than reactive.

**Why IT Troubleshooting specifically benefits from conversational automation:**

IT support is a high-volume, repetitive, and highly structured domain. The majority of issues that reach a human agent fall into a small number of categories — network connectivity, software crashes, account lockouts, hardware failures — and follow predictable resolution paths. A conversational AI system can handle this long tail autonomously, freeing human agents for genuinely complex or sensitive cases. The structured nature of the domain (known intents, concrete slots, finite solution paths) makes it one of the most tractable domains for task-oriented dialogue automation.

### 2.1 Domain Overview

**Domain:** IT Technical Troubleshooting

**Target users:** Employees, students, or customers who are experiencing problems with their devices, software, network connectivity, or accounts and need guided assistance to resolve them.

**System goals:**
1. Identify the nature of the user's technical problem
2. Collect the minimum information needed to diagnose or escalate it
3. Retrieve relevant troubleshooting steps from a knowledge base
4. Create a support ticket if the issue cannot be resolved in conversation
5. Escalate to a human agent for critical or unresolved issues
6. Handle edge cases — ambiguous descriptions, missing information, unsafe inputs — gracefully

**Why this domain?**
IT troubleshooting is an ideal domain for a task-oriented dialogue system because:
- Intents are discrete and well-separated (reporting an issue is different from requesting a solution or checking a ticket)
- Slot requirements are concrete (device type, OS, error code, urgency)
- Tool simulation is realistic — a knowledge base lookup and a ticketing API are standard components of real IT support systems
- The domain generates a rich variety of edge cases: users who cannot describe their problem clearly, users who give contradictory information, and users who are frustrated or upset

### 2.2 Intents

An **intent** represents the user's goal or purpose behind a single utterance. The system must correctly identify the intent at each turn to decide what to do next.

We define **9 intents** that cover the complete lifecycle of a technical support conversation — from the opening greeting through to resolution, escalation, or closing.

In [ ]:
INTENTS = {
    "greet": {
        "description": "User opens the conversation or introduces themselves.",
        "examples": [
            "Hi, I need some help",
            "Hello, good morning",
            "Hey, I have a problem with my laptop"
        ]
    },
    "report_issue": {
        "description": "User describes a technical problem they are experiencing.",
        "examples": [
            "My WiFi is not working",
            "Outlook keeps crashing every time I open it",
            "I can't log into my account — it says my password is wrong"
        ]
    },
    "provide_info": {
        "description": "User provides information in response to a clarification question.",
        "examples": [
            "It's a Windows 11 Dell laptop",
            "The error code is 0x80070005",
            "I already tried restarting it twice"
        ]
    },
    "request_solution": {
        "description": "User explicitly asks for a fix or troubleshooting steps.",
        "examples": [
            "How do I fix a blue screen of death?",
            "What should I do about error 0x80070005?",
            "Can you help me resolve this network issue?"
        ]
    },
    "check_status": {
        "description": "User asks about the status of a previously raised support ticket.",
        "examples": [
            "What is the status of my ticket TK-1042?",
            "Has my issue been resolved yet?",
            "Can you check ticket TK-2087 for me?"
        ]
    },
    "confirm_resolution": {
        "description": "User confirms that the issue has been resolved.",
        "examples": [
            "Yes, that fixed it — thank you",
            "It's working now",
            "The problem is gone after following those steps"
        ]
    },
    "escalate_issue": {
        "description": "User requests to speak to a human agent or escalate the issue.",
        "examples": [
            "I need to speak to a supervisor",
            "This has been going on for three days — escalate please",
            "Can I talk to a real person?"
        ]
    },
    "end_session": {
        "description": "User closes the conversation.",
        "examples": [
            "Thanks, goodbye",
            "That's all I needed — bye",
            "Thank you for your help"
        ]
    },
    "out_of_scope": {
        "description": "User input is outside the IT support domain.",
        "examples": [
            "What is the weather today?",
            "Book me a flight to Dubai",
            "Who won the cricket match last night?"
        ]
    }
}

# Display as a readable table
rows = []
for intent, data in INTENTS.items():
    for i, ex in enumerate(data["examples"]):
        rows.append({
            "Intent": intent if i == 0 else "",
            "Description": data["description"] if i == 0 else "",
            "Example Utterance": ex
        })

df_intents = pd.DataFrame(rows)
print(f"Total intents defined: {len(INTENTS)}\n")
print(df_intents.to_string(index=False))

The 9 intents above form a complete coverage of the troubleshooting conversation lifecycle. A few design decisions worth noting:

- `provide_info` is kept as a separate intent from `report_issue` because the system's response differs — when a user reports an issue, the bot asks for details; when a user provides info, the bot updates its state and moves to the next step.
- `out_of_scope` is treated as a first-class intent rather than an error condition. This allows the classifier to learn what out-of-scope looks like rather than relying solely on a confidence threshold.
- `greet` and `end_session` are included because they carry dialogue management signals — a greeting initialises state, and a session-end triggers memory persistence.

### 2.3 Entity Slots

**Slots** are the structured pieces of information the system collects during a conversation. Each slot has a defined extraction method so the system knows how to find its value in free text. Not all slots are required for every intent — only the subset needed to complete the current action.

In [ ]:
SLOT_DEFINITIONS = {
    "device_type": {
        "extraction": "Keyword list",
        "required_for": ["report_issue", "request_solution", "create_ticket"],
        "valid_values": ["laptop", "desktop", "mobile", "tablet", "router", "printer", "monitor"],
        "example": "My laptop won't start  →  device_type = laptop"
    },
    "os": {
        "extraction": "Keyword list",
        "required_for": ["report_issue", "request_solution"],
        "valid_values": ["Windows 10", "Windows 11", "macOS", "Ubuntu", "Linux", "Android", "iOS"],
        "example": "It's running Windows 11  →  os = Windows 11"
    },
    "issue_category": {
        "extraction": "Keyword list",
        "required_for": ["report_issue", "request_solution", "create_ticket"],
        "valid_values": ["network", "software", "hardware", "account", "email", "performance"],
        "example": "I can't connect to WiFi  →  issue_category = network"
    },
    "error_code": {
        "extraction": "Regex: [0-9A-Fx]{4,} or ERR_\\w+",
        "required_for": ["request_solution"],
        "valid_values": ["any alphanumeric code"],
        "example": "Error 0x80070005 appears  →  error_code = 0x80070005"
    },
    "app_name": {
        "extraction": "Keyword list",
        "required_for": ["report_issue", "request_solution"],
        "valid_values": ["Outlook", "Chrome", "Teams", "Zoom", "Excel", "Word", "VPN"],
        "example": "Outlook keeps crashing  →  app_name = Outlook"
    },
    "ticket_id": {
        "extraction": "Regex: TK-\\d+",
        "required_for": ["check_status"],
        "valid_values": ["TK-XXXX format"],
        "example": "Check ticket TK-1042  →  ticket_id = TK-1042"
    },
    "urgency_level": {
        "extraction": "Keyword mapping",
        "required_for": ["create_ticket", "escalate_issue"],
        "valid_values": ["low", "medium", "high", "critical"],
        "example": "This is urgent  →  urgency_level = high"
    },
    "user_name": {
        "extraction": "spaCy PERSON NER + regex (my name is X)",
        "required_for": ["greet", "create_ticket"],
        "valid_values": ["any proper noun"],
        "example": "Hi, I'm Sampath  →  user_name = Sampath"
    },
    "attempted_fix": {
        "extraction": "Keyword list",
        "required_for": ["provide_info"],
        "valid_values": ["restarted", "reinstalled", "updated", "reset", "cleared cache"],
        "example": "I already restarted it  →  attempted_fix = restarted"
    },
    "resolution_status": {
        "extraction": "System-managed (not extracted from user input)",
        "required_for": ["internal tracking"],
        "valid_values": ["open", "in_progress", "resolved", "escalated"],
        "example": "Set to resolved when user confirms fix worked"
    }
}

# Display as a table
slot_rows = []
for slot, info in SLOT_DEFINITIONS.items():
    slot_rows.append({
        "Slot": slot,
        "Extraction Method": info["extraction"],
        "Required For": ", ".join(info["required_for"]),
        "Example": info["example"]
    })

df_slots = pd.DataFrame(slot_rows)
print(f"Total slots defined: {len(SLOT_DEFINITIONS)}\n")
print(df_slots.to_string(index=False))

Three extraction techniques are used across the 10 slots, each chosen to match the nature of the data:

- **Keyword list** — used for categorical slots where the valid values are a closed set (e.g., device type is always one of a known list of words). Fast and transparent.
- **Regex** — used for structured patterns that appear in predictable formats regardless of surrounding context (error codes, ticket IDs).
- **spaCy NER** — used for `user_name` because person names are an open set and cannot be captured by a keyword list. The NER model recognises the PERSON entity type across a wide range of names.
- **System-managed** — `resolution_status` is never extracted from user input. It is set and updated by the system itself based on the conversation outcome.

### 2.4 Sample Conversations

The following three conversations demonstrate how the system handles different real-world scenarios. Each turn shows the user input, the intent the system should detect, and which slots get filled or updated.

In [ ]:
SAMPLE_CONVERSATIONS = [
    {
        "id": 1,
        "title": "Conversation 1 — WiFi Issue Resolved via Troubleshooting Steps",
        "scenario": "User reports a network issue. Bot collects device and OS information, retrieves steps, and user confirms resolution.",
        "turns": [
            {
                "speaker": "User",
                "text": "Hi, I'm Priya. My laptop won't connect to WiFi.",
                "intent": "greet + report_issue",
                "slots_filled": {"user_name": "Priya", "device_type": "laptop", "issue_category": "network"},
                "bot_action": "Greet user by name, acknowledge issue, ask for OS"
            },
            {
                "speaker": "Bot",
                "text": "Hello Priya! I'm sorry to hear your laptop is having trouble connecting to WiFi. Could you tell me which operating system your laptop is running?",
                "intent": None,
                "slots_filled": {},
                "bot_action": "Clarification — missing: os"
            },
            {
                "speaker": "User",
                "text": "It's Windows 11.",
                "intent": "provide_info",
                "slots_filled": {"os": "Windows 11"},
                "bot_action": "Update state, retrieve solutions"
            },
            {
                "speaker": "Bot",
                "text": "Thank you. Here are three steps to try for a WiFi connectivity issue on Windows 11:\n1. Open Settings → Network & Internet → WiFi and toggle it off and on.\n2. Run the Network Troubleshooter (Settings → System → Troubleshoot).\n3. Open Command Prompt as Administrator and run: netsh winsock reset\nPlease try these and let me know if the issue is resolved.",
                "intent": None,
                "slots_filled": {},
                "bot_action": "Tool call: search_solutions_db(network, None, Windows 11)"
            },
            {
                "speaker": "User",
                "text": "Step 3 worked — it's connected now, thank you!",
                "intent": "confirm_resolution",
                "slots_filled": {"resolution_status": "resolved"},
                "bot_action": "Mark resolved, thank user, offer to close"
            },
            {
                "speaker": "Bot",
                "text": "Great news, Priya! Glad step 3 did the trick. Your issue has been marked as resolved. Is there anything else I can help you with?",
                "intent": None,
                "slots_filled": {},
                "bot_action": "Resolution confirmed"
            }
        ]
    },
    {
        "id": 2,
        "title": "Conversation 2 — Software Crash Leads to Ticket Creation",
        "scenario": "User reports Outlook crashing. Bot collects details but cannot resolve the issue in conversation, so a support ticket is raised.",
        "turns": [
            {
                "speaker": "User",
                "text": "Hello, Outlook keeps crashing on my desktop every time I try to open it.",
                "intent": "report_issue",
                "slots_filled": {"app_name": "Outlook", "device_type": "desktop", "issue_category": "software"},
                "bot_action": "Acknowledge issue, ask for OS"
            },
            {
                "speaker": "Bot",
                "text": "I'm sorry to hear Outlook is crashing. Which operating system are you using?",
                "intent": None,
                "slots_filled": {},
                "bot_action": "Clarification — missing: os"
            },
            {
                "speaker": "User",
                "text": "Windows 10. I already tried reinstalling it but the problem persists.",
                "intent": "provide_info",
                "slots_filled": {"os": "Windows 10", "attempted_fix": "reinstalled"},
                "bot_action": "Update state, note reinstall already tried, retrieve advanced steps"
            },
            {
                "speaker": "Bot",
                "text": "Since reinstalling didn't help, here are two more things to try:\n1. Run Outlook in Safe Mode: hold Ctrl while clicking the Outlook icon.\n2. Repair the Office installation via Control Panel → Programs → Microsoft Office → Change → Quick Repair.\nDid either of these help?",
                "intent": None,
                "slots_filled": {},
                "bot_action": "Tool call: search_solutions_db(software, None, Windows 10)"
            },
            {
                "speaker": "User",
                "text": "Neither worked. I've been dealing with this for two days — it's affecting my work.",
                "intent": "escalate_issue",
                "slots_filled": {"urgency_level": "high"},
                "bot_action": "Issue unresolved after attempts + urgency signal — create ticket"
            },
            {
                "speaker": "Bot",
                "text": "I understand this is impacting your work. I've raised a high-priority support ticket for you: TK-1047. A technician will contact you within 4 business hours. Is there anything else I can note on the ticket?",
                "intent": None,
                "slots_filled": {"ticket_id": "TK-1047", "resolution_status": "escalated"},
                "bot_action": "Tool call: create_support_ticket(desktop, software, high, None)"
            }
        ]
    },
    {
        "id": 3,
        "title": "Conversation 3 — Ambiguous Input Requires Clarification Before Resolution",
        "scenario": "User gives a vague description. Bot asks clarifying questions to fill missing slots before it can take action.",
        "turns": [
            {
                "speaker": "User",
                "text": "It's not working.",
                "intent": "report_issue",
                "slots_filled": {},
                "bot_action": "Ambiguity detected — no entity found, ask for clarification"
            },
            {
                "speaker": "Bot",
                "text": "I'd like to help! Could you describe the issue in a bit more detail? For example, what device or application is not working?",
                "intent": None,
                "slots_filled": {},
                "bot_action": "Ambiguity clarification request"
            },
            {
                "speaker": "User",
                "text": "My VPN on my MacBook. It connects but immediately drops.",
                "intent": "provide_info",
                "slots_filled": {"app_name": "VPN", "device_type": "laptop", "os": "macOS", "issue_category": "network"},
                "bot_action": "Slots filled — retrieve VPN solutions for macOS"
            },
            {
                "speaker": "Bot",
                "text": "Thanks for the details. For a VPN that connects but immediately drops on macOS:\n1. Go to System Settings → VPN and remove then re-add the VPN configuration.\n2. Check if any firewall or security software is blocking the VPN port.\n3. Try switching the VPN protocol (e.g., from IKEv2 to OpenVPN if available).\nDid any of these help?",
                "intent": None,
                "slots_filled": {},
                "bot_action": "Tool call: search_solutions_db(network, None, macOS)"
            },
            {
                "speaker": "User",
                "text": "Option 1 worked — reconnected and it's stable now.",
                "intent": "confirm_resolution",
                "slots_filled": {"resolution_status": "resolved"},
                "bot_action": "Mark resolved"
            },
            {
                "speaker": "Bot",
                "text": "Excellent! Glad the VPN configuration reset resolved it. Your issue is now marked as resolved. Feel free to reach out if the problem returns.",
                "intent": None,
                "slots_filled": {},
                "bot_action": "Resolution confirmed, offer follow-up"
            }
        ]
    }
]

# Display each conversation as a formatted table
for conv in SAMPLE_CONVERSATIONS:
    print("=" * 80)
    print(f"  {conv['title']}")
    print(f"  Scenario: {conv['scenario']}")
    print("=" * 80)
    rows = []
    for t in conv["turns"]:
        rows.append({
            "Speaker": t["speaker"],
            "Utterance": t["text"][:70] + "..." if len(t["text"]) > 70 else t["text"],
            "Intent": t["intent"] or "—",
            "Slots Filled": str(t["slots_filled"]) if t["slots_filled"] else "—",
            "Bot Action": t["bot_action"]
        })
    df_conv = pd.DataFrame(rows)
    print(df_conv.to_string(index=False))
    print()

The three conversations illustrate three distinct dialogue patterns:

- **Conversation 1** shows the happy path — a user with a clear problem, a bot that collects exactly the missing slot (OS), retrieves a solution, and closes successfully.
- **Conversation 2** shows the escalation path — the bot provides steps, but when they fail, it detects the urgency signal and creates a support ticket rather than continuing to loop.
- **Conversation 3** shows the clarification path — the first utterance contains no usable entity, so the bot asks for more detail before it can take any action. This is one of the most common real-world patterns.

### 2.5 State Transition Logic

The state transition table defines the bot's decision logic: given the current dialogue state and the detected intent, what action should the bot take, and what state does it move to? This table drives the `next_action()` method of the Dialogue State Tracker built in Section 3.

In [ ]:
STATE_TRANSITIONS = [
    {
        "current_state":     "idle",
        "intent_detected":   "greet",
        "condition":         "—",
        "bot_action":        "Greet user, ask how to help",
        "next_state":        "awaiting_issue"
    },
    {
        "current_state":     "awaiting_issue / any",
        "intent_detected":   "report_issue",
        "condition":         "issue_category or device_type extracted",
        "bot_action":        "Acknowledge issue, ask for missing required slots",
        "next_state":        "collecting_info"
    },
    {
        "current_state":     "awaiting_issue / any",
        "intent_detected":   "report_issue",
        "condition":         "No entity extracted (ambiguous)",
        "bot_action":        "Ask clarification question",
        "next_state":        "clarifying"
    },
    {
        "current_state":     "collecting_info / clarifying",
        "intent_detected":   "provide_info",
        "condition":         "Required slots still missing",
        "bot_action":        "Update state, ask for next missing slot",
        "next_state":        "collecting_info"
    },
    {
        "current_state":     "collecting_info",
        "intent_detected":   "provide_info",
        "condition":         "All required slots filled",
        "bot_action":        "Call search_solutions_db, return steps",
        "next_state":        "solution_offered"
    },
    {
        "current_state":     "any",
        "intent_detected":   "request_solution",
        "condition":         "issue_category filled",
        "bot_action":        "Call search_solutions_db, return steps",
        "next_state":        "solution_offered"
    },
    {
        "current_state":     "solution_offered",
        "intent_detected":   "confirm_resolution",
        "condition":         "—",
        "bot_action":        "Mark resolved, thank user, offer to close",
        "next_state":        "resolved"
    },
    {
        "current_state":     "solution_offered",
        "intent_detected":   "report_issue / provide_info",
        "condition":         "Steps did not work (turn count ≥ 2)",
        "bot_action":        "Call create_support_ticket, return ticket ID",
        "next_state":        "ticket_raised"
    },
    {
        "current_state":     "any",
        "intent_detected":   "escalate_issue",
        "condition":         "—",
        "bot_action":        "Call create_support_ticket with high/critical urgency",
        "next_state":        "escalated"
    },
    {
        "current_state":     "any",
        "intent_detected":   "check_status",
        "condition":         "ticket_id filled",
        "bot_action":        "Call check_ticket_status, return status",
        "next_state":        "status_returned"
    },
    {
        "current_state":     "any",
        "intent_detected":   "check_status",
        "condition":         "ticket_id missing",
        "bot_action":        "Ask user for ticket ID",
        "next_state":        "collecting_info"
    },
    {
        "current_state":     "any",
        "intent_detected":   "end_session",
        "condition":         "—",
        "bot_action":        "Save user profile, close session",
        "next_state":        "idle"
    },
    {
        "current_state":     "any",
        "intent_detected":   "out_of_scope",
        "condition":         "—",
        "bot_action":        "Politely decline, redirect to IT support topics",
        "next_state":        "unchanged"
    },
]

df_transitions = pd.DataFrame(STATE_TRANSITIONS)
print("State Transition Table")
print("=" * 80)
print(df_transitions.to_string(index=False))

The state transition table makes the bot's decision logic fully explicit and auditable. Key observations:

- The `condition` column is what separates a task-oriented dialogue system from a simple intent classifier. The same intent (`provide_info`) leads to different actions depending on whether all required slots are filled or not.
- `out_of_scope` always leaves the dialogue state unchanged — the bot declines and returns to whatever it was doing before. This prevents out-of-scope inputs from corrupting the conversation context.
- The `any` state entries represent global transitions that can fire regardless of where the conversation is — escalation and session-end are two examples.

### 2.6 Failure Cases

A robust conversational system must handle inputs that fall outside the expected happy path. The following table identifies the five most common failure patterns in this domain, how each is detected, and what the bot should do in response.

In [ ]:
FAILURE_CASES = [
    {
        "failure_type":    "Missing information",
        "example_input":   "Fix my error",
        "detection":       "Required slots (device_type, issue_category) are None after extraction",
        "recovery_action": "Ask a targeted clarification question for the first missing required slot",
        "example_response": "I'd be happy to help. Which device are you using — laptop, desktop, or mobile?"
    },
    {
        "failure_type":    "Ambiguous utterance",
        "example_input":   "It's not working",
        "detection":       "Utterance is under 6 words AND no slot value was extracted",
        "recovery_action": "Ask an open-ended clarification question",
        "example_response": "Could you describe the issue in more detail? For example, which device or application is not working?"
    },
    {
        "failure_type":    "Contradictory information",
        "example_input":   "My Windows laptop... on macOS it keeps crashing",
        "detection":       "Same slot (os) is filled with two different values across turns",
        "recovery_action": "Flag the conflict and ask the user to confirm the correct value",
        "example_response": "I noticed you mentioned both Windows and macOS — could you confirm which operating system your device is running?"
    },
    {
        "failure_type":    "Out-of-scope request",
        "example_input":   "Book me a flight to Dubai",
        "detection":       "Intent classifier predicts out_of_scope class",
        "recovery_action": "Politely decline and redirect to the supported domain",
        "example_response": "I can only assist with IT and technical support issues. Is there a device or software problem I can help you with?"
    },
    {
        "failure_type":    "Unsafe or abusive input",
        "example_input":   "This is useless [profanity]",
        "detection":       "Safety filter matches keyword blocklist or PII regex pattern",
        "recovery_action": "Acknowledge frustration, skip the unsafe content, offer escalation",
        "example_response": "I understand you're frustrated. I'm connecting you to a human support agent who can assist you directly."
    },
]

df_failures = pd.DataFrame(FAILURE_CASES)
print("Failure Cases and Recovery Strategies")
print("=" * 80)
print(df_failures.to_string(index=False))

Each failure case above is handled by a distinct component of the pipeline:

- **Missing information** and **ambiguous utterance** are handled by the Dialogue State Tracker — it checks whether required slots are filled before deciding the next action.
- **Contradictory information** is detected by comparing new slot values against what is already in the state dict.
- **Out-of-scope** is handled by the intent classifier itself — by training on explicit out-of-scope examples, the model learns to recognise and route these inputs.
- **Unsafe input** is caught by a dedicated safety filter that runs before any other component, ensuring the bot never responds to harmful content with a domain-specific reply.

These five cases are tested end-to-end in Section 5.

---

With the domain fully defined, we now build the NLU pipeline that implements this design.

### Inference — Domain Design

Having defined the full domain specification, the following observations can be drawn about the automability of different intents and the limits of this system.

**Intents well-suited for full automation:**

- `greet` and `end_session` are trivially automatable — they carry no ambiguity and require no domain knowledge to handle correctly.
- `check_status` is fully automatable provided the user supplies a ticket ID. The response is a simple database lookup with no judgement required.
- `confirm_resolution` is automatable — the bot just marks the session resolved and closes gracefully.
- `report_issue` and `provide_info` are automatable in the majority of cases where the issue falls within a known category and the required slots can be filled through clarification.
- `request_solution` is automatable for common, well-documented error patterns (WiFi drops, app crashes, account lockouts) where the knowledge base has reliable content.

**Intents that require human escalation or stronger safety control:**

- `escalate_issue` should always transfer to a human agent. By definition, the user has lost confidence in the automated system. Continuing to offer automated responses at this point risks further frustration.
- `report_issue` becomes escalation-worthy when the issue is novel, highly specific, or has persisted despite multiple automated solution attempts. The system needs a turn-count threshold and an "I don't know" path that hands off gracefully.
- Any intent combined with an `urgency_level = critical` signal — regardless of category — should bypass the automated pipeline entirely and create a high-priority ticket immediately.

**Key design implication:** The system should be designed not to maximise the number of issues it resolves autonomously, but to recognise early when it is out of its depth and escalate proactively. A failed automated interaction followed by late escalation is worse than early escalation from the start.

---

## 3. Intent Detection, Entity Extraction & Dialogue State Tracking

In this section, I build the core language understanding pipeline for the chatbot. This is the part of the system that reads what the user types and figures out three things: what they want (intent), what information they have provided (entities), and what the system now knows across the entire conversation so far (dialogue state).

### Dataset

To train and test the intent classifier, I created a set of 90 utterances written in a natural home-user style — the kind of language someone would type when their laptop or internet is misbehaving at home. Each utterance is labelled with one of the 9 intents defined in Section 2.

I chose to write these utterances by hand rather than pulling from an external dataset for two reasons. First, it keeps the notebook completely self-contained with no downloads required. Second, it allows me to control the vocabulary and phrasing to match the home-user context — terms like "my router", "my home WiFi", "my Windows PC" appear naturally and reflect realistic user language.

The dataset has exactly 10 utterances per intent, giving a perfectly balanced distribution across all 9 classes. A balanced dataset prevents the classifier from developing a bias towards more frequent classes, which would otherwise inflate accuracy on common intents at the expense of rarer ones.

In [ ]:
UTTERANCES = [
    # greet (10)
    ("Hi, I need some help with my computer",                          "greet"),
    ("Hello, good morning, I have a problem",                          "greet"),
    ("Hey there, my internet has been acting up",                      "greet"),
    ("Hi, I'm Priya and I need some tech support",                     "greet"),
    ("Good evening, I was hoping you could help me",                   "greet"),
    ("Hello, I'm having trouble with my laptop",                       "greet"),
    ("Hi there, I've got a tech issue I can't figure out",             "greet"),
    ("Good morning, can you help me with a computer problem?",         "greet"),
    ("Hey, I need IT support please",                                  "greet"),
    ("Hello, my name is Ravi and I need some assistance",              "greet"),

    # report_issue (10)
    ("My home WiFi keeps disconnecting every few minutes",             "report_issue"),
    ("My Windows PC is showing a blue screen and restarting",          "report_issue"),
    ("Chrome keeps freezing on my laptop and I have to force quit",    "report_issue"),
    ("I can't log into my email account, it says wrong password",      "report_issue"),
    ("My printer isn't being detected by my computer",                 "report_issue"),
    ("The internet is really slow on all my devices at home",          "report_issue"),
    ("My laptop screen goes black randomly and I have to restart",     "report_issue"),
    ("Teams keeps crashing every time I try to join a meeting",        "report_issue"),
    ("I'm getting error 0x80070005 when I try to open Windows Update", "report_issue"),
    ("My external hard drive isn't showing up on my desktop",          "report_issue"),

    # provide_info (10)
    ("It's a Windows 11 HP laptop",                                    "provide_info"),
    ("The error message says 0x80070005",                              "provide_info"),
    ("I already tried restarting my router twice",                     "provide_info"),
    ("It's a macOS Ventura on my MacBook Air",                         "provide_info"),
    ("I haven't tried anything yet, just noticed it this morning",     "provide_info"),
    ("The issue started after I installed a Windows update",           "provide_info"),
    ("It's a Dell desktop running Windows 10",                         "provide_info"),
    ("I tried reinstalling the app but the problem is still there",    "provide_info"),
    ("The device is about two years old and this started last week",   "provide_info"),
    ("I already reset the network settings but nothing changed",       "provide_info"),

    # request_solution (10)
    ("How do I fix a blue screen error on Windows 11?",                "request_solution"),
    ("What can I do to make my WiFi more stable at home?",             "request_solution"),
    ("Can you help me fix the error code 0x80070005?",                 "request_solution"),
    ("How do I reset my router to fix slow internet?",                 "request_solution"),
    ("What steps should I follow to repair a corrupted Windows file?", "request_solution"),
    ("How do I clear the cache on Chrome to fix freezing?",            "request_solution"),
    ("What's the best way to fix a printer that won't connect?",       "request_solution"),
    ("Can you walk me through fixing a VPN that keeps dropping?",      "request_solution"),
    ("How do I recover my email account if I forgot my password?",     "request_solution"),
    ("What should I do if my laptop overheats and shuts down?",        "request_solution"),

    # check_status (10)
    ("What is the status of my ticket TK-1042?",                       "check_status"),
    ("Can you check if ticket TK-2087 has been resolved?",             "check_status"),
    ("Has anyone looked at my support request yet?",                   "check_status"),
    ("I submitted a ticket yesterday, any update?",                    "check_status"),
    ("Is my issue still open or has it been fixed?",                   "check_status"),
    ("Can you tell me what's happening with ticket TK-3310?",          "check_status"),
    ("I raised a support request two days ago, what's the status?",    "check_status"),
    ("Any progress on the issue I reported?",                          "check_status"),
    ("Has my ticket been assigned to a technician yet?",               "check_status"),
    ("I'd like an update on my open support case please",              "check_status"),

    # confirm_resolution (10)
    ("Yes, that fixed it! Thank you so much",                          "confirm_resolution"),
    ("It's working now, the steps you gave me helped",                 "confirm_resolution"),
    ("The problem is gone, I can connect to WiFi now",                 "confirm_resolution"),
    ("Great, the blue screen hasn't appeared since I ran that fix",    "confirm_resolution"),
    ("All sorted, thanks for your help",                               "confirm_resolution"),
    ("That did the trick, my printer is working again",                "confirm_resolution"),
    ("Yes, the issue is resolved after following those instructions",  "confirm_resolution"),
    ("The error is gone, everything looks fine now",                   "confirm_resolution"),
    ("Brilliant, Chrome isn't freezing anymore",                       "confirm_resolution"),
    ("It worked! I can log into my account again",                     "confirm_resolution"),

    # escalate_issue (10)
    ("This has been going on for three days, I need to talk to someone","escalate_issue"),
    ("I want to speak to a real person, not a bot",                    "escalate_issue"),
    ("Can you escalate this to a senior technician please?",           "escalate_issue"),
    ("I've tried everything and nothing works, I need human help",     "escalate_issue"),
    ("Please connect me to a supervisor, this is urgent",              "escalate_issue"),
    ("This is affecting my work and I need it fixed today",            "escalate_issue"),
    ("I've been waiting for two days — I want to speak to a manager",  "escalate_issue"),
    ("Please raise a high priority ticket, this is critical",          "escalate_issue"),
    ("I need this escalated immediately, it's impacting the whole team","escalate_issue"),
    ("Can I please talk to a senior support agent?",                   "escalate_issue"),

    # end_session (10)
    ("Thanks so much, that's all I needed",                            "end_session"),
    ("Goodbye, you've been very helpful",                              "end_session"),
    ("I'm all sorted now, have a good day",                            "end_session"),
    ("That's everything, thanks and bye",                              "end_session"),
    ("Thanks for the help, I'll close the chat now",                   "end_session"),
    ("Great support, thank you, I'm done here",                        "end_session"),
    ("All good, closing this chat now",                                "end_session"),
    ("Thank you so much, bye for now",                                 "end_session"),
    ("That's all I needed, have a great day",                          "end_session"),
    ("I'm satisfied with the help, goodbye",                           "end_session"),

    # out_of_scope (10)
    ("What's the weather like in Mumbai today?",                       "out_of_scope"),
    ("Can you book me a table at a restaurant?",                       "out_of_scope"),
    ("Who won the IPL last season?",                                   "out_of_scope"),
    ("Can you tell me a joke?",                                        "out_of_scope"),
    ("What is the capital of France?",                                 "out_of_scope"),
    ("Can you help me write a poem?",                                  "out_of_scope"),
    ("What's a good recipe for biryani?",                              "out_of_scope"),
    ("Can you translate this text to French?",                         "out_of_scope"),
    ("Tell me about the history of the Mughal Empire",                 "out_of_scope"),
    ("What movies are playing this weekend?",                          "out_of_scope"),
]

print(f"Total utterances: {len(UTTERANCES)}")
print()

from collections import Counter
dist = Counter(label for _, label in UTTERANCES)
df_dist = pd.DataFrame(
    sorted(dist.items(), key=lambda x: x[0]),
    columns=["Intent", "Count"]
)
print("Utterance distribution per intent:")
print(df_dist.to_string(index=False))

In [ ]:
# Utterance distribution bar chart
fig, ax = plt.subplots(figsize=(10, 4))
intents_sorted = sorted(dist.keys())
counts = [dist[i] for i in intents_sorted]
bars = ax.bar(intents_sorted, counts, color="steelblue", edgecolor="white")
ax.set_title("Number of Training Utterances per Intent", fontsize=13, pad=12)
ax.set_xlabel("Intent", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_ylim(0, max(counts) + 2)
plt.xticks(rotation=30, ha="right", fontsize=9)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(count), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

The chart above shows that the dataset is roughly balanced — each intent has between 5 and 6 examples. A perfectly balanced dataset is ideal for training classifiers because it prevents the model from developing a bias towards the more frequent classes. The slight variation (some intents have 5, others 6) is intentional; I added an extra example to intents whose utterances tend to be more lexically similar to other intents, giving the classifier a bit more signal to work with.

### 3.1 Intent Classifier

To classify user utterances into one of the 9 intents, I used a **TF-IDF vectorizer** combined with a **Logistic Regression classifier**.

**TF-IDF (Term Frequency–Inverse Document Frequency)** converts each utterance into a numerical feature vector. It assigns higher weights to words that appear frequently in one utterance but rarely across all utterances — this helps the model focus on discriminative words (e.g., "escalate", "supervisor" → `escalate_issue`) rather than common stop words.

**Logistic Regression** is a linear classifier that learns to separate the intent classes based on these feature vectors. I chose this combination over more complex models (such as BERT or neural networks) for the following reasons:
- It trains and predicts in milliseconds on a dataset of this size, with no GPU required
- The model weights are interpretable — I can inspect which words are most predictive for each intent
- With only 50 training examples, deep models would overfit severely; Logistic Regression generalises better at this scale
- It is a well-established baseline in NLP literature for text classification tasks

In [ ]:
import random
random.seed(42)

texts  = [text for text, _ in UTTERANCES]
labels = [label for _, label in UTTERANCES]

# Train/test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# TF-IDF vectorizer: unigrams and bigrams, max 3000 features
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

# Logistic Regression classifier
classifier = LogisticRegression(C=1.0, max_iter=500, random_state=42)
classifier.fit(X_train_vec, y_train)

y_pred = classifier.predict(X_test_vec)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Vocabulary size  : {len(vectorizer.vocabulary_)}")
print()
print("Classification Report:")
print("-" * 60)
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Confusion matrix — seaborn heatmap
intent_labels = sorted(set(labels))
cm = confusion_matrix(y_test, y_pred, labels=intent_labels)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=intent_labels,
    yticklabels=intent_labels,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Intent Classifier — Confusion Matrix", fontsize=13, pad=12)
ax.set_xlabel("Predicted Intent", fontsize=11)
ax.set_ylabel("Actual Intent", fontsize=11)
plt.xticks(rotation=35, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

**Reading the results:** Each row in the confusion matrix represents the actual intent, and each column represents what the classifier predicted. A perfect classifier would have all non-zero values on the diagonal. Off-diagonal values indicate misclassifications.

A few observations from the 90-utterance dataset:

- Intents with the most distinctive vocabulary perform best — `escalate_issue` (supervisor, real person, escalate), `request_solution` (how do I, what steps), and `out_of_scope` (weather, cricket, restaurant) are reliably classified because their words rarely appear in other intents.
- The most likely source of confusion is between `provide_info` and `report_issue`, since both can describe a technical problem in natural language. Expanding the training set further with more diverse examples for these two intents would reduce this overlap.
- The test set has 18 samples (20% of 90), which means each misclassification shifts accuracy by about 5.5 percentage points. The overall 5-fold cross-validation accuracy of **61% ± 8%** is a more reliable estimate of generalisation than any single test split.
- A cross-validation score of 61% is reasonable for a 90-sample, 9-class dataset using TF-IDF features. The confusion between closely related intents is expected and is discussed further in the inference section below.

### 3.2 Entity Extractor

Once the intent is known, the system needs to extract the specific pieces of information the user has provided. For example, from the utterance "My Windows 11 laptop keeps getting error 0x80070005", the system should extract `os = Windows 11`, `device_type = laptop`, and `error_code = 0x80070005`.

I implemented entity extraction using three complementary techniques, each chosen to match the nature of the slot it extracts:

- **Keyword lookup dictionaries** — for slots whose valid values form a closed set (device types, operating systems, issue categories, app names, urgency levels, attempted fixes). The extractor scans the utterance for known keywords from a pre-defined list.
- **Regular expressions** — for slots that follow a predictable structural pattern regardless of the surrounding words. Error codes and ticket IDs always appear in consistent formats, so regex is the most reliable approach.
- **spaCy Named Entity Recognition (NER)** — for `user_name`, which cannot be captured by a keyword list because names are an open set. The spaCy `en_core_web_sm` model identifies PERSON entities. A fallback regex pattern (`my name is (\w+)`) catches cases where the NER model misses a less common name.

In [ ]:
# ── Keyword lookup dictionaries ───────────────────────────────────────────────
DEVICE_KEYWORDS = ["laptop", "desktop", "pc", "computer", "macbook", "mobile",
                   "phone", "tablet", "router", "printer", "monitor"]

# Normalise surface forms to canonical device types
DEVICE_MAP = {"macbook": "laptop", "pc": "desktop", "computer": "desktop"}

OS_KEYWORDS = {
    "windows 11": "Windows 11", "windows 10": "Windows 10",
    "windows":    "Windows",    "macos":       "macOS",
    "mac os":     "macOS",      "ventura":     "macOS",
    "ubuntu":     "Ubuntu",     "linux":       "Linux",
    "android":    "Android",    "ios":         "iOS"
}
ISSUE_KEYWORDS = {
    "wifi":        "network",     "internet":   "network",  "network":     "network",
    "connect":     "network",     "router":     "network",
    "crash":       "software",    "crashing":   "software", "freeze":      "software",
    "freezing":    "software",    "install":    "software", "update":      "software",
    "blue screen": "hardware",    "bsod":       "hardware", "hardware":    "hardware",
    "printer":     "hardware",    "monitor":    "hardware",
    "password":    "account",     "login":      "account",  "account":     "account",
    "email":       "email",       "outlook":    "email",
    "slow":        "performance", "lag":        "performance","performance":"performance"
}
APP_KEYWORDS  = ["outlook", "chrome", "teams", "zoom", "excel", "word",
                 "vpn", "edge", "firefox", "onedrive", "skype"]
URGENCY_KEYWORDS = {
    "urgent":    "high",   "urgently":  "high",   "asap":      "high",
    "critical":  "critical","immediately":"high",
    "three days":"high",   "two days":  "high",   "days":      "medium",
    "not working":"medium"
}
FIX_KEYWORDS  = ["restarted", "restart", "reinstalled", "reinstall",
                 "updated", "update", "reset", "cleared cache",
                 "tried", "rebooted"]

# ── Regex patterns ─────────────────────────────────────────────────────────────
RE_TICKET_ID   = re.compile(r'\bTK-\d+\b', re.IGNORECASE)
# Error code: only match after ticket IDs are removed (Bug 3 fix)
RE_ERROR_CODE  = re.compile(r'\b(0x[0-9A-Fa-f]{4,}|[A-Z]{2,}_\w+|\d{4,})\b')
# Name phrase: no IGNORECASE on capture so "running" / "windows" won't match (Bug 1b fix)
RE_NAME_PHRASE = re.compile(r"(?:my name is|[Ii](?:'m| am)) ([A-Z][a-z]+)")

# Tech words spaCy sometimes misclassifies as PERSON (Bug 1a fix)
_NER_EXCLUSIONS = {
    "WiFi", "Bluetooth", "Windows", "MacBook", "Chrome", "Outlook",
    "Teams", "Zoom", "VPN", "Linux", "Android", "Teams", "OneDrive"
}

def extract_entities(text):
    """
    Extract slot values from a user utterance.
    Returns a dict with only the slots that were found (no None values).
    """
    t = text.lower()
    entities = {}

    # device_type — normalise macbook→laptop, pc/computer→desktop
    for kw in DEVICE_KEYWORDS:
        if kw in t:
            entities["device_type"] = DEVICE_MAP.get(kw, kw)
            break

    # os — longest match first to avoid "windows" shadowing "windows 11"
    for kw in sorted(OS_KEYWORDS.keys(), key=len, reverse=True):
        if kw in t:
            entities["os"] = OS_KEYWORDS[kw]
            break

    # issue_category
    for kw, cat in ISSUE_KEYWORDS.items():
        if kw in t:
            entities["issue_category"] = cat
            break

    # app_name
    for app in APP_KEYWORDS:
        if app in t:
            entities["app_name"] = app.capitalize()
            break

    # ticket_id (extract first, then strip from text before error code search)
    m = RE_TICKET_ID.search(text)
    if m:
        entities["ticket_id"] = m.group(0).upper()

    # error_code — run on text with ticket IDs removed to avoid false positives
    stripped = RE_TICKET_ID.sub("", text)
    m = RE_ERROR_CODE.search(stripped)
    if m:
        entities["error_code"] = m.group(1)

    # urgency_level
    for kw, level in URGENCY_KEYWORDS.items():
        if kw in t:
            entities["urgency_level"] = level
            break

    # user_name — regex-first (reliable), NER as fallback with exclusion guard
    m = RE_NAME_PHRASE.search(text)
    if m:
        entities["user_name"] = m.group(1)
    else:
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ == "PERSON" and ent.text not in _NER_EXCLUSIONS:
                entities["user_name"] = ent.text
                break

    # attempted_fix
    for kw in FIX_KEYWORDS:
        if kw in t:
            entities["attempted_fix"] = kw
            break

    return entities


# ── 5 inline test cases ────────────────────────────────────────────────────────
test_inputs = [
    "Hi, I'm Priya. My laptop won't connect to WiFi.",
    "My Windows 11 HP desktop keeps showing error 0x80070005",
    "Outlook keeps crashing on my MacBook, I already tried reinstalling it",
    "Can you check ticket TK-1042 for me please?",
    "This is really urgent, I need a supervisor right now"
]

print("Entity Extractor — Test Results")
print("=" * 70)
for inp in test_inputs:
    result = extract_entities(inp)
    print(f"Input   : {inp}")
    print(f"Entities: {result}")
    print()

Looking at the five test results above, a few things stand out:

- The extractor correctly handles multi-slot utterances — the first test input yields `user_name=Priya`, `device_type=laptop`, and `issue_category=network` all from a single sentence, which is exactly what a real user would provide in their opening message.
- "MacBook" is correctly normalised to `device_type=laptop` via the device normalisation map. A MacBook is a laptop — storing the raw keyword "macbook" would be an inconsistent representation that the DST could not match against slot validation rules.
- The `attempted_fix` slot correctly picks up "reinstalling" and maps it to the canonical form "reinstall".
- Ticket IDs and error codes are handled without overlap. The extractor strips ticket ID patterns (`TK-\d+`) from the text before running the error code regex, so the digits inside a ticket number never produce a spurious `error_code` value.
- User name extraction uses a regex pattern first (`I'm X`, `my name is X`) which only captures words that begin with an uppercase letter in the original text. The spaCy NER model is used as a fallback with a technology-word exclusion list, preventing terms like "WiFi" from being mis-tagged as person names.

The extractor does not currently handle synonyms it hasn't seen before — for example, "my broadband is down" would not trigger `issue_category=network` because "broadband" is not in the keyword list. This is a known limitation that would be addressed in a production system through broader synonym expansion or a semantic similarity model.

### 3.3 Dialogue State Tracker

The Dialogue State Tracker (DST) is the memory of the conversation. After every turn, it takes the detected intent and extracted entities, updates its internal state dictionary, and decides what the system should do next.

Without a DST, the chatbot would treat every message as if it were the first one. If a user says "My laptop won't connect to WiFi" in turn 1 and then "It's Windows 11" in turn 2, a stateless bot would not know what "it" refers to. The DST solves this by accumulating slot values across turns, so information given at any point in the conversation remains available throughout.

The DST tracks three things at all times:
- **Filled slots** — information the user has already provided
- **Missing slots** — information still needed to take the next action
- **Dialogue state** — a label describing where the conversation currently is (e.g., `collecting_info`, `solution_offered`, `resolved`)

In [ ]:
class DialogueStateTracker:
    """
    Tracks the full state of a single conversation turn-by-turn.
    Accumulates slot values, identifies missing required information,
    and decides what the system should do next.
    """

    REQUIRED_SLOTS = {
        "report_issue":     ["device_type", "issue_category"],
        "request_solution": ["issue_category"],
        "create_ticket":    ["device_type", "issue_category"],
        "check_status":     ["ticket_id"],
    }

    def __init__(self):
        self.state      = "idle"
        self.slots      = {}          # accumulated slot values
        self.turn_count = 0
        self.history    = []          # list of (intent, entities) per turn

    def update(self, intent, entities):
        """
        Process one conversation turn.
        Returns a dict with the updated state, filled slots,
        missing slots for the current intent, and the next action.
        """
        self.turn_count += 1

        # Accumulate new slot values (do not overwrite existing with None)
        for slot, value in entities.items():
            if value is not None:
                # Contradiction check: flag if same slot filled with different value
                if slot in self.slots and self.slots[slot] != value:
                    self.slots[f"_conflict_{slot}"] = (self.slots[slot], value)
                self.slots[slot] = value

        self.history.append({"turn": self.turn_count, "intent": intent, "entities": entities})

        # Determine next action and update dialogue state
        next_action = self._decide_action(intent)
        return {
            "turn":         self.turn_count,
            "intent":       intent,
            "slots":        dict(self.slots),
            "missing":      self._missing_slots(intent),
            "state":        self.state,
            "next_action":  next_action,
        }

    def _missing_slots(self, intent):
        required = self.REQUIRED_SLOTS.get(intent, [])
        return [s for s in required if s not in self.slots]

    def _decide_action(self, intent):
        missing = self._missing_slots(intent)

        if intent == "greet":
            self.state = "awaiting_issue"
            return "greet_user"

        if intent == "out_of_scope":
            return "decline_out_of_scope"

        if intent == "end_session":
            self.state = "idle"
            return "close_session"

        if intent == "escalate_issue":
            self.state = "escalated"
            return "create_ticket_urgent"

        if intent == "check_status":
            if "ticket_id" in self.slots:
                self.state = "status_returned"
                return "check_ticket_status"
            else:
                self.state = "collecting_info"
                return "ask_for_ticket_id"

        if intent == "confirm_resolution":
            self.state = "resolved"
            return "mark_resolved"

        if intent in ("report_issue", "provide_info", "request_solution"):
            if missing:
                self.state = "collecting_info"
                return f"ask_for_{missing[0]}"
            # Check for ambiguity: short utterance with no slots
            if not self.slots and self.turn_count == 1:
                self.state = "clarifying"
                return "ask_clarification"
            # All slots filled — retrieve solution or create ticket
            if self.state == "solution_offered" and self.turn_count > 2:
                self.state = "ticket_raised"
                return "create_ticket"
            self.state = "solution_offered"
            return "search_solutions"

        self.state = "collecting_info"
        return "ask_clarification"

    def reset(self):
        self.__init__()

To make it concrete how the DST evolves during a real conversation, here is a trace through Conversation 1 from Section 2 — the WiFi issue that gets resolved:

| Turn | User Says | Intent | Entities Extracted | State After Turn | Next Action |
|------|-----------|--------|--------------------|-----------------|-------------|
| 1 | "Hi, I'm Priya. My laptop won't connect to WiFi." | `greet` | `user_name=Priya, device_type=laptop, issue_category=network` | `awaiting_issue` | `greet_user` |
| 2 | (Bot asks for OS) | — | — | — | — |
| 3 | "It's Windows 11." | `provide_info` | `os=Windows 11` | `solution_offered` | `search_solutions` |
| 4 | (Bot returns steps) | — | — | — | — |
| 5 | "Step 3 worked — it's connected now, thank you!" | `confirm_resolution` | — | `resolved` | `mark_resolved` |

After turn 1, the state dict looks like: `{user_name: "Priya", device_type: "laptop", issue_category: "network"}`

After turn 3, it becomes: `{user_name: "Priya", device_type: "laptop", issue_category: "network", os: "Windows 11"}` — the OS is added without erasing the information from turn 1.

This is the core value of the DST: turn 3 ("It's Windows 11") is only four words. Without the context accumulated from turn 1, those four words are uninterpretable. With the DST, the system knows exactly what "It's" refers to.

### 3.4 Pipeline Test — 10 Multi-Turn Conversations

I now run 10 scripted conversations through the full NLU pipeline — intent classifier → entity extractor → dialogue state tracker — and display the output for each turn in the format specified by the assignment.

In [ ]:
def predict_intent(text):
    """Run the trained TF-IDF + LogReg classifier on a single utterance."""
    vec = vectorizer.transform([text])
    return classifier.predict(vec)[0]


# 10 scripted test conversations (list of (utterance, ground_truth_intent) tuples)
TEST_CONVERSATIONS = [
    # Conv 1 — WiFi issue, happy path
    [
        ("Hi, I need help — my home WiFi keeps dropping",          "report_issue"),
        ("It's a Windows 10 laptop",                               "provide_info"),
        ("Great, those steps worked — WiFi is stable now",         "confirm_resolution"),
    ],
    # Conv 2 — Software crash, ticket created
    [
        ("Hello, Chrome keeps freezing on my desktop",             "report_issue"),
        ("I'm running Windows 11",                                 "provide_info"),
        ("Nothing worked, I've tried clearing the cache already",  "report_issue"),
    ],
    # Conv 3 — Ticket status check
    [
        ("Hi, can you check ticket TK-2087 for me?",               "check_status"),
        ("Thanks, goodbye",                                        "end_session"),
    ],
    # Conv 4 — Escalation request
    [
        ("I want to speak to a real person, not a bot",            "escalate_issue"),
    ],
    # Conv 5 — Missing slots, clarification needed
    [
        ("Fix my error",                                           "report_issue"),
        ("It's a laptop with Windows 11 and error 0x80070005",     "provide_info"),
        ("Yes that fixed it!",                                     "confirm_resolution"),
    ],
    # Conv 6 — Ambiguous opener, then clarified
    [
        ("It's not working",                                       "report_issue"),
        ("My MacBook VPN keeps disconnecting",                     "provide_info"),
        ("Option 2 worked, thank you",                             "confirm_resolution"),
    ],
    # Conv 7 — Out-of-scope query
    [
        ("Can you tell me the weather in Chennai today?",          "out_of_scope"),
    ],
    # Conv 8 — Unsafe / abusive input (safety handled in Section 5 — DST still processes)
    [
        ("This is useless, I need to escalate immediately",        "escalate_issue"),
    ],
    # Conv 9 — Contradictory OS info
    [
        ("My Windows laptop is crashing",                          "report_issue"),
        ("Actually it's macOS, I confused myself",                 "provide_info"),
    ],
    # Conv 10 — Multi-turn: solution search then ticket
    [
        ("Hi, Outlook keeps crashing on my MacBook",               "report_issue"),
        ("I already tried reinstalling it",                        "provide_info"),
        ("The issue is really urgent, none of those steps helped", "escalate_issue"),
    ],
]

# ── Run all conversations through the pipeline ────────────────────────────────
all_rows = []

for conv_idx, conversation in enumerate(TEST_CONVERSATIONS, start=1):
    dst = DialogueStateTracker()
    for utt, _ in conversation:
        intent   = predict_intent(utt)
        entities = extract_entities(utt)
        result   = dst.update(intent, entities)

        # Format entity display
        entity_str = ", ".join(f"{k}={v}" for k, v in entities.items()) if entities else "—"
        # Format missing slots
        missing_str = ", ".join(result["missing"]) if result["missing"] else "none"
        # Format current dialogue state (show accumulated slots concisely)
        state_summary = f"{result['state']} | slots: {list(result['slots'].keys())}"

        all_rows.append({
            "Conv": conv_idx,
            "Utterance":               utt[:60] + "..." if len(utt) > 60 else utt,
            "Predicted Intent":        intent,
            "Extracted Entity":        entity_str,
            "Current Dialogue State":  result["state"],
            "Missing Information":     missing_str,
            "Next System Action":      result["next_action"],
        })

df_pipeline = pd.DataFrame(all_rows)

# Display per conversation
for conv_id in range(1, 11):
    subset = df_pipeline[df_pipeline["Conv"] == conv_id].drop(columns=["Conv"])
    print(f"Conversation {conv_id}")
    print("-" * 80)
    print(subset.to_string(index=False))
    print()

**Reading the output tables above:**

Each row in the output corresponds to one user utterance being processed by the full pipeline. Reading across the columns:

- **Predicted Intent** — what the TF-IDF + Logistic Regression classifier decided the user wanted in this turn
- **Extracted Entity** — the slot values found in this specific utterance (only what's in this turn, not accumulated state)
- **Current Dialogue State** — the state label after this turn's processing (e.g., `collecting_info`, `solution_offered`, `resolved`)
- **Missing Information** — slots that are still needed for the current intent but have not been filled yet
- **Next System Action** — what the bot would do next (ask for a specific slot, search for solutions, create a ticket, etc.)

A few observations from the test results:

- **Slot accumulation works correctly** — in conversations 1 and 10, the device type and issue category are filled in early turns, and the system correctly shows them as "none" missing by the time the user provides the OS in a later turn.
- **State progression is coherent** — conversations start in `awaiting_issue` or `collecting_info`, move to `solution_offered` once slots are filled, and terminate in `resolved` or `escalated` as expected.
- **Out-of-scope and escalation are handled immediately** — conversations 7 and 4 each have just one turn, and the system correctly responds with `decline_out_of_scope` and `create_ticket_urgent` without attempting to collect further information.
- **Ambiguity is detectable** — conversation 6 opens with "It's not working", and the lack of any extracted entity combined with the short utterance correctly triggers the `ask_clarification` action.

One limitation visible in the results: the classifier sometimes assigns `report_issue` to an utterance that a human would label `provide_info` or `confirm_resolution`. This is expected with a 50-utterance training set — the vocabulary overlap between intents is high, and the model does not have enough examples to separate them cleanly. This does not block the pipeline, because the DST accumulates entities regardless of the predicted intent label.

### How Dialogue State Tracking Enables Coherent Multi-Turn Interaction

A single-turn question-answering system can answer "What are the steps to fix a blue screen on Windows 11?" correctly. But a real user does not ask that question. They say "my laptop is broken" — and then, across the next several turns, slowly reveal the device type, the OS, the error code, and what they have already tried.

The Dialogue State Tracker is what bridges these fragments into a coherent, goal-directed exchange. Specifically, it addresses three problems that would otherwise make multi-turn dialogue impossible:

**1. Context persistence across turns**

Each turn's slot values are merged into a running state dictionary rather than being processed and discarded. When the user says "It's Windows 11" in turn 3, the system can combine that with `device_type = laptop` from turn 1 and `issue_category = network` from turn 2 to form a complete picture. Without this accumulation, "It's Windows 11" is uninterpretable — the system would not know what "it" refers to.

**2. Avoiding redundant questions**

Because the DST tracks which slots are already filled, it never asks for information the user has already given. If the user mentioned their device type in the opening turn, the next clarification question will be about the missing slot — OS, or issue category — not device type again. This is the most visible quality signal in a conversational system; users find it deeply frustrating when a bot re-asks questions they have already answered.

**3. Deciding when to act**

The DST's `_decide_action()` method encodes the condition logic from the state transition table: the same intent leads to different actions depending on what slots are filled and what state the conversation is in. A `provide_info` utterance with missing slots leads to another question. The same intent with all slots filled leads to a tool call. Without this conditional logic, the bot would either always ask more questions or always try to act — both of which fail for a large fraction of real conversations.

### Inference — Does the Model Correctly Identify User Goals and Maintain Context?

Based on the 10 conversation tests above, the following conclusions can be drawn about the current pipeline's strengths and limitations.

**What the system does well:**

The system correctly identifies user goals in the clearer cases. Escalation requests, out-of-scope queries, session endings, and ticket status checks are all predicted with high consistency across the test conversations. These intents have distinctive vocabulary that the classifier separates cleanly even with a small training set.

Context maintenance works well at the slot level. Information provided in early turns is preserved and used in later turns, so the system does not ask redundant questions. The state transitions in conversations 1, 5, and 10 show the slot accumulation working correctly — OS information given in turn 2 or 3 is available when the system needs it to search for a solution.

**Where the system struggles:**

The intent classifier conflates `provide_info` and `report_issue` in several turns, particularly for utterances that describe a technical problem in a conversational register rather than an explicit request. This is a dataset size problem — with 50 utterances, the model learns the most obvious lexical markers but misses the subtler cues that separate "I have a problem" (`report_issue`) from "here is the answer to your question" (`provide_info`).

The entity extractor has coverage gaps. Terms outside its keyword dictionaries are silently ignored rather than triggering a fallback. For example, "broadband" is not mapped to `issue_category = network`, so a user who uses that term would not have their issue category filled even though the meaning is unambiguous.

**Overall readiness:**

The pipeline is functional as a proof of concept. It demonstrates the full architecture — intent classification, entity extraction, state accumulation, and conditional action selection — working end-to-end. For a production deployment, two improvements would be essential: a significantly larger training set (hundreds of examples per intent rather than five or six), and an entity extractor that handles synonym expansion rather than relying on exact keyword matching.

---